# 02 · Energy Analysis ⭐

This is the core analysis notebook for the thesis. It provides a deep dive into CPU and
Memory energy consumption across 18 languages, grouped by execution compiler (AOT, JIT,
Interpreted).

**Units:** All energy values are in **Joules (J)** (converted from raw µJ at load time).

**Key questions:**
- Which languages are most energy-efficient?
- Do compilers (AOT vs JIT vs Interpreted) differ significantly in energy consumption?
- How does energy vary across benchmarks?

**Methodology:**
- Rankings, heatmaps and summary tables use the **two-step mean** (mean per
  language × benchmark, then averaged across the 8 benchmarks with equal weight),
  sourced from `results_clean_runs.csv` via `lang_means()`.
- Significance testing keeps **non-parametric** tests, which are robust to the
  right-skewed, non-normal benchmark distributions:
  - **Kruskal-Wallis** (non-parametric ANOVA) for compiler comparisons
  - **Mann-Whitney U** (pairwise) with **Bonferroni correction** for post-hoc tests
  - **Rank-biserial correlation** as the effect size measure
- Violin plots are shown as distribution views (median and mean lines marked),
  one per execution model; the rankings and the heatmap use the two-step mean.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

import importlib
import plot_style as ps
importlib.reload(ps)   # pick up edits to plot_style.py without a kernel restart
ps.apply_style()

# Canonical constants — single source: plot_style.
COL_CPU_ENERGY, COL_MEM_ENERGY = ps.COL_CPU_ENERGY, ps.COL_MEM_ENERGY
COL_TIME                       = ps.COL_TIME
COL_CPU_CARBON, COL_MEM_CARBON = ps.COL_CPU_CARBON, ps.COL_MEM_CARBON
COMPILER        = ps.COMPILER
COMPILER_COLORS = ps.COMPILER_COLORS
COMPILER_ORDER  = ps.COMPILER_ORDER
MEANPROPS       = ps.MEANPROPS
ALPHA           = ps.ALPHA

OUTPUTS_DIR = Path('outputs'); OUTPUTS_DIR.mkdir(exist_ok=True)

# Single source of truth: per-run rows (df) + per-cell means with EDP (df_mean).
df      = ps.load_runs()
df_mean = ps.cell_means(df)

def lang_means(cols):
    """Per-language two-step mean (equal benchmark weight) for column(s) `cols`."""
    return ps.lang_means(df_mean, cols)

print(f"Runs: {df.shape} | Cell-means: {df_mean.shape} | "
      f"{df['language'].nunique()} languages \u00d7 {df['benchmark'].nunique()} benchmarks")
df_mean.head(3)

## 1. Execution Model Comparison

**Violin plots** show the full distribution shape per compiler.
**Kruskal-Wallis** tests whether any compiler differs significantly.
If significant, **pairwise Mann-Whitney U** tests with **Bonferroni correction** identify which pairs differ.
Effect size is reported as **rank-biserial correlation** r = 1 − 2U/(n₁·n₂).

In [ ]:
import matplotlib.ticker as mticker

# Violin distribution of CPU energy per execution model on a LOG scale — energy
# spans ~3 orders of magnitude, so the data is log10-transformed before the KDE
# (violinplot computes its KDE in linear space, so set_yscale('log') would distort
# the shape) and the y-ticks are relabelled back to real Joules.
fig, ax = plt.subplots(figsize=(7, 6))
groups = [np.log10(df[df['compiler'] == p][COL_CPU_ENERGY].values) for p in COMPILER_ORDER]
parts  = ax.violinplot(groups, positions=range(len(COMPILER_ORDER)), showmedians=True, showmeans=True)
for pc, p in zip(parts['bodies'], COMPILER_ORDER):
    pc.set_facecolor(COMPILER_COLORS[p])
    pc.set_alpha(0.85)
for key in ('cmedians', 'cmeans', 'cbars', 'cmins', 'cmaxes'):
    if key in parts:
        parts[key].set_edgecolor('#333333')
        parts[key].set_linewidth(1)
ax.set_xticks(range(len(COMPILER_ORDER)))
ax.set_xticklabels(COMPILER_ORDER)
ax.yaxis.set_major_locator(mticker.MultipleLocator(1))              # one tick per decade
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{10 ** v:g}'))
ax.set_ylabel('CPU Energy (J, log scale)')
plt.tight_layout()
ps.save_fig(fig, '02_cpu_energy_violin_compiler')
plt.show()

In [ ]:
import matplotlib.ticker as mticker

# Violin distribution of memory (DRAM) energy per execution model on a LOG scale,
# same technique as the CPU violin: log10-transform the data before the KDE and
# relabel the y-ticks back to real Joules.
fig, ax = plt.subplots(figsize=(7, 6))
groups = [np.log10(df[df['compiler'] == p][COL_MEM_ENERGY].values) for p in COMPILER_ORDER]
parts  = ax.violinplot(groups, positions=range(len(COMPILER_ORDER)), showmedians=True, showmeans=True)
for pc, p in zip(parts['bodies'], COMPILER_ORDER):
    pc.set_facecolor(COMPILER_COLORS[p])
    pc.set_alpha(0.85)
for key in ('cmedians', 'cmeans', 'cbars', 'cmins', 'cmaxes'):
    if key in parts:
        parts[key].set_edgecolor('#333333')
        parts[key].set_linewidth(1)
ax.set_xticks(range(len(COMPILER_ORDER)))
ax.set_xticklabels(COMPILER_ORDER)
ax.yaxis.set_major_locator(mticker.MultipleLocator(1))              # one tick per decade
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{10 ** v:g}'))
ax.set_ylabel('Memory Energy (J, log scale)')
plt.tight_layout()
ps.save_fig(fig, '02_mem_energy_violin_compiler')
plt.show()

> **Takeaway:** the violins confirm the compiler gap; Kruskal-Wallis (below) tests whether it is statistically significant.

In [ ]:
def rank_biserial(x, y):
    """Rank-biserial correlation as effect size for Mann-Whitney U."""
    u, _ = stats.mannwhitneyu(x, y, alternative='two-sided')
    return 1 - (2 * u) / (len(x) * len(y))

for col, label in [(COL_CPU_ENERGY, 'CPU Energy (J)'), (COL_MEM_ENERGY, 'Memory Energy (J)')]:
    groups   = {p: df[df['compiler'] == p][col].values for p in COMPILER_ORDER}
    kw_stat, kw_p = stats.kruskal(*groups.values())
    n_pairs = len(COMPILER_ORDER) * (len(COMPILER_ORDER) - 1) // 2

    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"  Kruskal-Wallis H={kw_stat:.3f}, p={kw_p:.4f} ", end='')
    print("(SIGNIFICANT)" if kw_p < ALPHA else "(not significant)")

    if kw_p < ALPHA:
        print(f"  Post-hoc Mann-Whitney U (Bonferroni α={ALPHA/n_pairs:.4f}):")
        for (p1, p2) in combinations(COMPILER_ORDER, 2):
            u, p = stats.mannwhitneyu(groups[p1], groups[p2], alternative='two-sided')
            p_adj = min(p * n_pairs, 1.0)
            r = rank_biserial(groups[p1], groups[p2])
            sig = "✓" if p_adj < ALPHA else "✗"
            print(f"    {sig} {p1} vs {p2}: U={u:.0f}, p_adj={p_adj:.4f}, r={r:.3f}")

## 2. Per-Benchmark Energy Heatmap

Heatmap of mean CPU and Memory energy (J) for each language × benchmark combination
(the per-cell means stored in `results_clean_runs.csv`). This reveals which benchmarks are
most energy-intensive and which languages suffer disproportionately on specific workloads.

In [ ]:
# Per-cell means come directly from df_mean (results_clean.csv).
pivot_cpu = df_mean.pivot(index='language', columns='benchmark', values=COL_CPU_ENERGY)
pivot_mem = df_mean.pivot(index='language', columns='benchmark', values=COL_MEM_ENERGY)

lang_sort = lang_means(COL_CPU_ENERGY).sort_values().index
pivot_cpu = pivot_cpu.loc[lang_sort]
pivot_mem = pivot_mem.loc[lang_sort]

fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot_cpu, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.3, cbar_kws={'label': 'Energy (J)'})
ax.set_xlabel('')
ax.set_ylabel('')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
ps.save_fig(fig, '02_cpu_energy_heatmap_benchmark')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot_mem, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
            linewidths=0.3, cbar_kws={'label': 'Energy (J)'})
ax.set_xlabel('')
ax.set_ylabel('')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
ps.save_fig(fig, '02_mem_energy_heatmap_benchmark')
plt.show()

> **Takeaway:** k-nucleotide and regex-redux are the most energy-intensive benchmarks, and the interpreted languages suffer most on them.

## 3. Energy Efficiency Ranking

Languages ranked by **mean** CPU energy and by **mean** memory energy (J), ascending
(two-step mean, equal benchmark weight) — most efficient at the top.

In [ ]:
# Per-language two-step mean CPU & memory energy (equal benchmark weight); the two
# ranking bar charts below order languages by these columns.
ranking = lang_means([COL_CPU_ENERGY, COL_MEM_ENERGY]).copy()
ranking.columns = ['cpu_mean_J', 'mem_mean_J']
ranking.insert(0, 'compiler', ranking.index.map(COMPILER))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ranked = ranking.sort_values('cpu_mean_J')   # order bars by the plotted metric
colors = [COMPILER_COLORS[COMPILER[l]] for l in ranked.index]
bars = ax.barh(ranked.index, ranked['cpu_mean_J'], color=colors, alpha=0.85, edgecolor='white')

# value label at the end of each bar
x_max = ranked['cpu_mean_J'].max()
for bar, val in zip(bars, ranked['cpu_mean_J']):
    ax.text(bar.get_width() + x_max * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,.2f}', va='center', ha='left', fontsize=8, color='#333333')
ax.set_xlim(0, x_max * 1.12)

ax.set_title('', fontsize=12)
ax.set_xlabel('Mean CPU Energy (J)')
ax.set_ylabel('')
ax.invert_yaxis()   # most efficient (lowest) at the top
legend_handles = [mpatches.Patch(color=COMPILER_COLORS[p], label=p, alpha=0.85)
                  for p in COMPILER_ORDER]
ax.legend(handles=legend_handles, title='Execution Model', loc='upper right')
plt.tight_layout()
ps.save_fig(fig, '02_cpu_energy_ranking')
plt.show()

> **Takeaway:** the final CPU-energy ranking is led by the AOT native compilers and trailed by the interpreted languages.

### Memory Energy Ranking

Same view for **mean memory (DRAM) energy (J)** — lower is more efficient. Ordered by the plotted metric, so the order differs from the CPU ranking.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ranked_mem = ranking.sort_values('mem_mean_J')   # order bars by the plotted metric
colors = [COMPILER_COLORS[COMPILER[l]] for l in ranked_mem.index]
bars = ax.barh(ranked_mem.index, ranked_mem['mem_mean_J'], color=colors, alpha=0.85, edgecolor='white')

# value label at the end of each bar
x_max = ranked_mem['mem_mean_J'].max()
for bar, val in zip(bars, ranked_mem['mem_mean_J']):
    ax.text(bar.get_width() + x_max * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,.2f}', va='center', ha='left', fontsize=8, color='#333333')
ax.set_xlim(0, x_max * 1.12)

ax.set_title('', fontsize=12)
ax.set_xlabel('Mean Memory Energy (J)')
ax.set_ylabel('')
ax.invert_yaxis()   # most efficient (lowest) at the top
legend_handles = [mpatches.Patch(color=COMPILER_COLORS[p], label=p, alpha=0.85)
                  for p in COMPILER_ORDER]
ax.legend(handles=legend_handles, title='Execution Model', loc='upper right')
plt.tight_layout()
ps.save_fig(fig, '02_mem_energy_ranking')
plt.show()

> **Takeaway:** memory energy is dominated by Erlang (driven by its very long regex-redux run); the rest cluster low, so the DRAM ranking differs markedly from the CPU-energy ranking.